# 🚀 Open in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matheus-rech/pitvqa-surgical-workflow/blob/main/notebooks/02_pitvqa_sage_training_prep.ipynb)

**IMPORTANT:** Run this notebook in Google Colab for best download speeds!

---

# 🏥 PitVQA → SAGE/Molmo Training Dataset

This notebook prepares the PitVQA dataset for fine-tuning SAGE/Molmo on pituitary surgery:

1. **Downloads** PitVQA from UCL (with resume support)
2. **Creates** SFT dataset in Molmo conversation format
3. **Pushes** to HuggingFace Hub for HF Skills training
4. **Generates** the HF Skills training prompt

**Output:** Ready-to-train dataset at `matheus-rech/pitvqa-sage-sft`

---
**Project:** PitVQA Surgical Workflow Understanding  
**Target:** MICCAI 2026

## 1. Setup & Dependencies

In [ ]:
# Install dependencies
%pip install -q huggingface_hub datasets pillow tqdm pandas numpy requests

# Install aria2 for faster downloads (Colab)
!apt-get install -y aria2 > /dev/null 2>&1 || echo "aria2 not available, using standard download"

In [ ]:
import os
import json
import zipfile
import shutil
import subprocess
from pathlib import Path
from typing import List, Dict, Optional
from tqdm.auto import tqdm
import numpy as np

# Detect environment
try:
    import google.colab
    IN_COLAB = True
    BASE_DIR = "/content"
except ImportError:
    IN_COLAB = False
    BASE_DIR = os.getcwd()

DOWNLOAD_DIR = os.path.join(BASE_DIR, "pitvqa_download")
DATA_DIR = os.path.join(BASE_DIR, "pitvqa_data")
OUTPUT_DIR = os.path.join(BASE_DIR, "pitvqa_sft")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📍 Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"📂 Download: {DOWNLOAD_DIR}")
print(f"📂 Data: {DATA_DIR}")
print(f"📂 Output: {OUTPUT_DIR}")

# Check disk space
total, used, free = shutil.disk_usage(BASE_DIR)
print(f"\n💾 Disk: {free // (1024**3)} GB free (need ~15GB)")

## 2. HuggingFace Authentication

In [ ]:
from huggingface_hub import login, HfApi

# Set your token here or login interactively
HF_TOKEN = ""  # Paste token here, or leave empty for interactive login

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()  # Interactive

# Configuration
HF_USERNAME = "matheus-rech"
RAW_DATASET = f"{HF_USERNAME}/pitvqa-surgical"  # Raw frames & annotations
SFT_DATASET = f"{HF_USERNAME}/pitvqa-sage-sft"  # SFT training format
OUTPUT_MODEL = f"{HF_USERNAME}/pitvqa-sage-surgical"  # Fine-tuned model

print(f"\n✅ Logged in to HuggingFace")
print(f"📦 Raw dataset: {RAW_DATASET}")
print(f"📦 SFT dataset: {SFT_DATASET}")
print(f"🤖 Output model: {OUTPUT_MODEL}")

## 3. Download PitVQA Dataset (Resumable)

Uses aria2c for faster, resumable downloads when available.

In [ ]:
import requests
import urllib.request

DOWNLOAD_URLS = {
    "videos": ("https://rdr.ucl.ac.uk/ndownloader/files/49158880", "8.11GB"),
    "annotations": ("https://rdr.ucl.ac.uk/ndownloader/files/49228108", "~50MB"),
    "frame_annotations": ("https://rdr.ucl.ac.uk/ndownloader/files/49228111", "~200MB")
}

def download_with_aria2(url: str, output_path: str) -> bool:
    """Fast download with aria2c (supports resume)."""
    try:
        result = subprocess.run([
            "aria2c", "-x", "16", "-s", "16", "-k", "1M",
            "--continue=true",
            "-d", os.path.dirname(output_path),
            "-o", os.path.basename(output_path),
            url
        ], capture_output=True, text=True)
        return result.returncode == 0
    except FileNotFoundError:
        return False

def download_with_resume(url: str, output_path: str) -> bool:
    """Download with resume support using requests."""
    headers = {}
    mode = 'wb'
    initial_pos = 0
    
    if os.path.exists(output_path):
        initial_pos = os.path.getsize(output_path)
        headers['Range'] = f'bytes={initial_pos}-'
        mode = 'ab'
    
    response = requests.get(url, headers=headers, stream=True)
    
    if response.status_code == 416:  # Range not satisfiable = complete
        return True
    
    total_size = int(response.headers.get('content-length', 0)) + initial_pos
    
    with open(output_path, mode) as f:
        with tqdm(total=total_size, initial=initial_pos, unit='B', unit_scale=True) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return True

def download_file(name: str, url: str, size_hint: str) -> str:
    """Download file with best available method."""
    output_path = os.path.join(DOWNLOAD_DIR, f"{name}.zip")
    
    # Check if complete
    if os.path.exists(output_path):
        try:
            with zipfile.ZipFile(output_path, 'r') as zf:
                zf.testzip()
            print(f"✅ {name}.zip already complete")
            return output_path
        except (zipfile.BadZipFile, Exception):
            print(f"⚠️ {name}.zip incomplete, resuming...")
    
    print(f"\n📥 Downloading {name} ({size_hint})...")
    
    # Try aria2c first (faster, resume-capable)
    if download_with_aria2(url, output_path):
        print(f"✅ Downloaded with aria2c: {name}")
        return output_path
    
    # Fallback to requests with resume
    if download_with_resume(url, output_path):
        print(f"✅ Downloaded: {name}")
        return output_path
    
    raise Exception(f"Failed to download {name}")

# Download all files
print("📥 Starting downloads...")
print("   This may take 30-60 minutes for the full dataset.\n")

downloaded = {}
for name, (url, size) in DOWNLOAD_URLS.items():
    downloaded[name] = download_file(name, url, size)

print("\n✅ All downloads complete!")

In [ ]:
# Verify downloads
print("🔍 Verifying downloads...\n")

for name, path in downloaded.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        try:
            with zipfile.ZipFile(path, 'r') as zf:
                zf.testzip()
            print(f"✅ {name}: {size_mb:.1f} MB - Valid")
        except zipfile.BadZipFile:
            print(f"❌ {name}: {size_mb:.1f} MB - INVALID (re-download needed)")
    else:
        print(f"❌ {name}: Missing")

## 4. Extract & Organize Data

In [ ]:
print("📦 Extracting files...\n")

for name, path in downloaded.items():
    if not os.path.exists(path):
        print(f"⚠️ Skipping {name} (not found)")
        continue
    
    extract_dir = os.path.join(DATA_DIR, name)
    
    # Check if already extracted
    if os.path.exists(extract_dir) and len(os.listdir(extract_dir)) > 1:
        print(f"✅ {name} already extracted")
        continue
    
    os.makedirs(extract_dir, exist_ok=True)
    
    print(f"📦 Extracting {name}...")
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            zf.extractall(extract_dir)
        print(f"   ✅ Extracted to {extract_dir}")
    except Exception as e:
        print(f"   ❌ Error: {e}")

print("\n✅ Extraction complete!")

In [ ]:
# Explore extracted structure
from pathlib import Path

data_path = Path(DATA_DIR)

# Find all frames and annotations
frames = list(data_path.rglob("*.jpg")) + list(data_path.rglob("*.png"))
json_files = list(data_path.rglob("*.json"))
csv_files = list(data_path.rglob("*.csv"))

print(f"🖼️  Found {len(frames)} frames")
print(f"📄 Found {len(json_files)} JSON files")
print(f"📄 Found {len(csv_files)} CSV files")

# Show sample paths
if frames:
    print(f"\nSample frames:")
    for f in frames[:3]:
        print(f"  {f}")

if json_files:
    print(f"\nJSON files:")
    for f in json_files[:5]:
        print(f"  {f}")

## 5. Create SFT Dataset for SAGE/Molmo Training

Converts PitVQA annotations to conversation format compatible with SAGE/Molmo fine-tuning.

In [ ]:
# PitVQA Surgical Vocabulary
PHASES = ["Nasal", "Sellar", "Tumor Removal", "Closure"]
STEPS = [
    "Septal Dissection", "Turbinectomy", "Sphenoidotomy",
    "Posterior Septectomy", "Sellar Floor Removal", "Dura Opening",
    "Tumor Resection", "Hemostasis", "Reconstruction", "Nasal Packing",
    "Visualization", "Instrument Change", "Suction", "Irrigation", "Other"
]
INSTRUMENTS = [
    "Endoscope", "Suction", "Curette", "Bipolar", "Monopolar",
    "Scissors", "Grasper", "Drill", "Kerrison", "Speculum",
    "Cottonoid", "Hemostatic Agent", "Fat Graft", "Fascia", "Nasoseptal Flap"
]
ANATOMICAL_STRUCTURES = [
    "Pituitary Gland", "Pituitary Tumor", "Carotid Artery", "Optic Nerve",
    "Sella Floor", "Sphenoid Sinus", "Dura Mater", "Nasal Septum"
]

# Question templates for surgical VQA
QUESTION_TEMPLATES = {
    "phase": [
        "What surgical phase is shown in this image?",
        "Which phase of the pituitary surgery is currently being performed?",
        "Identify the current surgical phase."
    ],
    "step": [
        "What surgical step is being performed?",
        "Describe the current surgical action.",
        "What procedure is the surgeon performing?"
    ],
    "instruments": [
        "What surgical instruments are visible in this image?",
        "Identify the tools being used.",
        "Which instruments can you see in the surgical field?"
    ],
    "pointing": [
        "Point to the {structure} in this image.",
        "Where is the {structure} located?",
        "Indicate the position of the {structure}."
    ]
}

print("✅ Loaded surgical vocabulary")
print(f"   Phases: {len(PHASES)}")
print(f"   Steps: {len(STEPS)}")
print(f"   Instruments: {len(INSTRUMENTS)}")
print(f"   Structures: {len(ANATOMICAL_STRUCTURES)}")

In [ ]:
def load_annotations() -> Dict:
    """Load PitVQA annotation files."""
    annotations = {}
    
    # Find annotation files
    ann_dir = Path(DATA_DIR)
    
    for json_file in ann_dir.rglob("*.json"):
        try:
            with open(json_file, 'r') as f:
                data = json.load(f)
            annotations[json_file.stem] = data
            print(f"✅ Loaded {json_file.name}")
        except Exception as e:
            print(f"⚠️ Could not load {json_file.name}: {e}")
    
    return annotations

annotations = load_annotations()
print(f"\n📄 Loaded {len(annotations)} annotation files")

In [ ]:
def create_sft_sample(frame_path: str, phase: str, step: str, 
                       instruments: List[str], qa_type: str) -> Dict:
    """Create a single SFT sample in Molmo conversation format."""
    
    templates = QUESTION_TEMPLATES[qa_type]
    question = np.random.choice(templates)
    
    if qa_type == "phase":
        answer = f"This image shows the {phase} phase of pituitary surgery."
    elif qa_type == "step":
        answer = f"The surgeon is performing {step}."
    elif qa_type == "instruments":
        if len(instruments) == 1:
            instr_str = instruments[0]
        else:
            instr_str = ", ".join(instruments[:-1]) + f" and {instruments[-1]}"
        answer = f"The visible instruments are: {instr_str}."
    elif qa_type == "pointing":
        structure = np.random.choice(ANATOMICAL_STRUCTURES)
        question = question.format(structure=structure)
        # Generate pseudo-random point (will be replaced with real annotations)
        x, y = np.random.uniform(0.3, 0.7), np.random.uniform(0.3, 0.7)
        answer = f"The {structure.lower()} is located here: <point x='{x:.2f}' y='{y:.2f}'>{structure.lower().replace(' ', '_')}</point>"
    else:
        return None
    
    return {
        "messages": [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ],
        "image": frame_path,
        "phase": phase.lower().replace(" ", "_"),
        "step": step.lower().replace(" ", "_"),
        "instruments": [i.lower().replace(" ", "_") for i in instruments],
        "qa_type": qa_type
    }

def generate_sft_dataset(frames: List[Path], annotations: Dict, 
                          max_samples: int = 50000) -> List[Dict]:
    """Generate SFT dataset from frames and annotations."""
    samples = []
    qa_types = ["phase", "step", "instruments", "pointing"]
    
    # If we have annotations, use them; otherwise generate synthetic labels
    use_synthetic = len(annotations) == 0
    if use_synthetic:
        print("⚠️ No annotations found, using synthetic labels for demo")
    
    for frame_path in tqdm(frames[:max_samples // 4], desc="Creating SFT samples"):
        # Get or generate annotations for this frame
        if use_synthetic:
            phase = np.random.choice(PHASES)
            step = np.random.choice(STEPS)
            num_instruments = np.random.randint(1, 4)
            instruments = list(np.random.choice(INSTRUMENTS, num_instruments, replace=False))
        else:
            # TODO: Extract from actual annotations based on frame
            frame_id = frame_path.stem
            phase = np.random.choice(PHASES)
            step = np.random.choice(STEPS)
            instruments = list(np.random.choice(INSTRUMENTS, np.random.randint(1, 4), replace=False))
        
        # Generate samples for each QA type
        for qa_type in qa_types:
            sample = create_sft_sample(
                str(frame_path), phase, step, instruments, qa_type
            )
            if sample:
                samples.append(sample)
    
    return samples

print("✅ SFT generation functions ready")

In [ ]:
# Generate SFT dataset
print("📊 Creating SFT dataset...\n")

if len(frames) == 0:
    print("⚠️ No frames found. Creating minimal demo dataset...")
    # Create demo samples without images for testing
    sft_samples = []
    for i in range(1000):
        phase = np.random.choice(PHASES)
        step = np.random.choice(STEPS)
        instruments = list(np.random.choice(INSTRUMENTS, np.random.randint(1, 4), replace=False))
        
        for qa_type in ["phase", "step", "instruments"]:
            sample = create_sft_sample(f"frame_{i:05d}.jpg", phase, step, instruments, qa_type)
            if sample:
                sft_samples.append(sample)
else:
    sft_samples = generate_sft_dataset(frames, annotations, max_samples=50000)

print(f"\n✅ Created {len(sft_samples)} SFT samples")

In [ ]:
# Show sample
if sft_samples:
    print("🔍 Sample SFT entry:\n")
    sample = sft_samples[0]
    print(json.dumps(sample, indent=2))

## 6. Create HuggingFace Dataset

In [ ]:
from datasets import Dataset, DatasetDict, Features, Value, Sequence, Image

# Create dataset
print("📊 Creating HuggingFace Dataset...\n")

# Convert to dataset format
dataset = Dataset.from_list(sft_samples)

# Split into train/val/test
split = dataset.train_test_split(test_size=0.2, seed=42)
val_test = split["test"].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    "train": split["train"],
    "validation": val_test["train"],
    "test": val_test["test"]
})

print(f"✅ Dataset created:")
print(f"   Train: {len(dataset_dict['train'])} samples")
print(f"   Validation: {len(dataset_dict['validation'])} samples")
print(f"   Test: {len(dataset_dict['test'])} samples")

In [ ]:
# Save locally
local_path = os.path.join(OUTPUT_DIR, "dataset")
dataset_dict.save_to_disk(local_path)
print(f"💾 Saved to: {local_path}")

## 7. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import create_repo

# Create repository if needed
try:
    create_repo(SFT_DATASET, repo_type="dataset", exist_ok=True)
    print(f"✅ Repository ready: {SFT_DATASET}")
except Exception as e:
    print(f"⚠️ {e}")

In [ ]:
# Push dataset to Hub
print(f"📤 Pushing to HuggingFace Hub...")
print(f"   Repository: https://huggingface.co/datasets/{SFT_DATASET}\n")

dataset_dict.push_to_hub(
    SFT_DATASET,
    private=False,
    commit_message="Upload PitVQA SFT dataset for SAGE/Molmo training"
)

print(f"\n🎉 Dataset pushed successfully!")
print(f"📦 https://huggingface.co/datasets/{SFT_DATASET}")

## 8. Generate HF Skills Training Prompt

This prompt can be used with Claude Code's HF Skills to start training on HuggingFace Jobs.

In [ ]:
HF_SKILLS_PROMPT = f"""Fine-tune allenai/SAGE-MM-Molmo2-8B-SFT_RL on {SFT_DATASET}

Configuration:
- Output model: {OUTPUT_MODEL}
- Epochs: 3
- Batch size: 4
- Learning rate: 2e-5
- Use LoRA: True (r=16, alpha=32)
- Hardware: a10g-large
- This is a vision language model with images in 'image' column
- Dataset has 'messages' column in conversation format

Training objective:
Fine-tune SAGE/Molmo to understand pituitary surgery videos:
- Recognize surgical phases (Nasal, Sellar, Tumor Removal, Closure)
- Identify surgical steps (15 different procedures)
- Detect instruments (18 surgical tools)
- Point to anatomical structures using <point> format
"""

print("=" * 60)
print("🎯 HF SKILLS TRAINING PROMPT")
print("=" * 60)
print(HF_SKILLS_PROMPT)
print("=" * 60)

# Save prompt
prompt_path = os.path.join(OUTPUT_DIR, "hf_skills_prompt.txt")
with open(prompt_path, 'w') as f:
    f.write(HF_SKILLS_PROMPT)
print(f"\n💾 Saved to: {prompt_path}")

## 9. Summary & Next Steps

In [ ]:
print("""
════════════════════════════════════════════════════════════
🎉 PIPELINE COMPLETE!
════════════════════════════════════════════════════════════

📊 What was created:
""")

print(f"   1. SFT Dataset: https://huggingface.co/datasets/{SFT_DATASET}")
print(f"      - {len(dataset_dict['train'])} training samples")
print(f"      - {len(dataset_dict['validation'])} validation samples")
print(f"      - {len(dataset_dict['test'])} test samples")
print(f"\n   2. HF Skills Prompt: {prompt_path}")

print("""
🚀 Next Steps:

   Option 1: Use HF Skills (Recommended)
   ─────────────────────────────────
   Copy the HF Skills prompt above and paste it into:
   - Claude Code (with HF Skills enabled)
   - HuggingFace Spaces with Claude integration
   
   The training will run on HuggingFace Jobs infrastructure
   with A10G GPUs - no local GPU required!

   Option 2: Manual Training
   ─────────────────────────────────
   Use the generated training script with TRL:
   
   from trl import SFTTrainer
   from datasets import load_dataset
   
   dataset = load_dataset("matheus-rech/pitvqa-sage-sft")
   # ... configure trainer

════════════════════════════════════════════════════════════
Project: PitVQA Surgical Workflow Understanding - MICCAI 2026
════════════════════════════════════════════════════════════
""")

## 10. Cleanup (Optional)

In [ ]:
# Cleanup to free disk space
cleanup = input("Delete local files to free disk space? (y/n): ")

if cleanup.lower() == 'y':
    shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
    print("✅ Cleaned up download directory")
    
    # Keep data for now
    # shutil.rmtree(DATA_DIR, ignore_errors=True)
else:
    print("⏭️ Keeping local files")

# Show disk usage
total, used, free = shutil.disk_usage(BASE_DIR)
print(f"\n💾 Disk: {free // (1024**3)} GB free")